## **Access file**

In [ ]:
from itertools import count

import mne


raw = mne.io.read_raw_gdf("data/BCICIV_2a_gdf/A01T.gdf", preload=True)
raw.plot()

## info of dataset

In [ ]:

print(raw.info)
events, event_dict = mne.events_from_annotations(raw)
print(events)
print(event_dict)


## count the number of event which is required

In [ ]:
import numpy as np

for event_id in [7, 8, 9, 10]:
    count = np.sum(events[:,2] == event_id)
    print(f"Event ID :{event_id} count: {count}")



# preprocessing

### Epochs


In [ ]:
event_id = {
    "Left_hand":7,
    "Right_hand":8,
    "feet":9,
    "tongue":10
}

epochs = mne.Epochs(
    raw,
    event_id=event_id,
    tmin=0,
    tmax=4,
    baseline=None,
    preload=True,
)

#print(epochs)
#print(epochs.get_data().shape)
#print(epochs.ch_names)

epochs.set_channel_types({
    "EOG-left":'eog',
    "EOG-central":'eog',
    "EOG-right":'eog',
})

eeg_epochs = epochs.copy().pick("eeg")
filtered_epochs = eeg_epochs.copy()
filtered_epochs.filter(                      #Band-pass filter (focus on 8hz-30hz)
    l_freq=8,
    h_freq=30,

)
print(filtered_epochs.get_data().shape)



## visualize filterd epoch and non-filtered epoch

In [ ]:
raw_signal = eeg_epochs.get_data()[0,0,:]
filtered_signal = filtered_epochs.get_data()[0,0,:]

import matplotlib.pyplot as plt
plt.figure(figsize=(12,4))

plt.plot(raw_signal, label="Raw")
plt.plot(filtered_signal, label="Filtered")
plt.xlabel('Time point')
plt.ylabel('Amplitude')
plt.title('Raw vs. Filtered waveform')
plt.legend()
plt.show()

from scipy.signal import welch

sfreq = eeg_epochs.info['sfreq']

freqs_raw, psd_raw = welch(
    raw_signal,
    fs=sfreq,
    nperseg=500,
)

freqs_filtered, psd_filtered = welch(
    filtered_signal,
    fs=sfreq,
    nperseg=500,
)

plt.figure(figsize=(10,4))
plt.semilogy(freqs_raw, psd_raw, label="Raw")
plt.semilogy(freqs_filtered, psd_filtered, label="Filtered")
plt.xlim(0,50)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density')
plt.title('Frequency spectrum Before and After filtering')
plt.legend()
plt.show()

# Train model using CSP + LDA

## split

In [ ]:
x = filtered_epochs.get_data()
y = filtered_epochs.events[:, 2]

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

#Conver the label
labelencoder = LabelEncoder()
y = labelencoder.fit_transform(y)

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'x_shape {x.shape}')
print(f'y_shape {y.shape}')
print(f"X_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {x_test.shape}, y_test: {y_test.shape}")

## Train and Test

In [ ]:
from mne.decoding import CSP
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

csp = CSP(
    n_components=10,
    reg=None,
    log=True,
    norm_trace=False
)

clf = Pipeline([
    ('csp', csp),
    ('lda', LinearDiscriminantAnalysis()),
])

clf.fit(x_train, y_train)
y_pred = clf.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.3f}')

cm = confusion_matrix(y_test, y_pred)
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labelencoder.classes_,
)
disp.plot()
plt.title('CSP + LDA Confusion Matrix')
plt.show()

## Cross- Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


components_list = [4,6,8,10]

for n in components_list:
    model = Pipeline([
        ('CSP', CSP(n_components= n, reg=None, log=True, norm_trace=False)),
        ("LDA", LinearDiscriminantAnalysis())
    ])

    scores = cross_val_score(
        model,
        x_train,
        y_train,
        cv=CV,
        scoring="accuracy",
    )
    print(f'{n} components')
    print(f'{np.mean(scores):.3f} +- {np.std(scores):.3f}')

# Train model using EEG-Net

## unsqueeze

In [ ]:
import torch

x_train_torch = torch.tensor(
    x_train,
    dtype=torch.float32
).unsqueeze(1)

x_test_torch = torch.tensor(
    x_test,
    dtype=torch.float32
).unsqueeze(1)

y_train_torch = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_torch = torch.tensor(
    y_test,
    dtype=torch.long
)




## train the model

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class EEGNet(nn.Module):

    def __init__(self,n_classes=4):
        super().__init__()

        self.temporal = nn.Sequential(
            nn.Conv2d(
            in_channels=1,
            out_channels=8,
            kernel_size=(1, 64),
            padding=(8, 32),
            bias=False,
            ), nn.BatchNorm2d(8)
        )

        self.spatial = nn.Sequential(
            nn.Conv2d(
                in_channels=8,
                out_channels=16,
                kernel_size=(22, 1),
                groups=8,
                bias=False,
            ),
            nn.BatchNorm2d(16),
            nn.ELU()
        )

        self.pool1 = nn.AvgPool2d(
            kernel_size=(1, 4),
        )

        self.dropout1 = nn.Dropout(p= 0.25)

        self.separable = nn.Sequential(
          nn.Conv2d(
            in_channels=16,
            out_channels=16,
            kernel_size=(1, 16),
            padding=(0, 8),
            groups=16,
            bias=False
        ),

        nn.Conv2d(
            in_channels=16,
            out_channels=16,
            kernel_size=(1, 1),
            bias=False
        ),

        nn.BatchNorm2d(16),
        nn.ELU()
        )


        self.pool2 = nn.AvgPool2d(
            kernel_size=(1, 8)
        )

        self.dropout2 = nn.Dropout(
            p=0.25
        )


        self.classifier = nn.LazyLinear(n_classes)


    def forward(self, x):
        x = self.temporal(x)


        x = self.spatial(x)


        x = self.pool1(x)
        x = self.dropout1(x)

        x = self.separable(x)

        x = self.pool2(x)
        x = self.dropout2(x)

        x = torch.flatten(x, start_dim=1)

        x = self.classifier(x)

        return x

model = EEGNet(n_classes=4)

print(model)

train_dataset = TensorDataset(x_train_torch, y_train_torch)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 50

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        # Forward
        outputs = model(X_batch)

        # Calculate loss
        loss = criterion(outputs, y_batch)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{num_epochs}], "
        f"Loss: {epoch_loss:.4f}"
    )

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

X_train_dl, X_val_dl, y_train_dl, y_val_dl = train_test_split(
    x_train_torch,
    y_train_torch,
    test_size=0.2,
    stratify=y_train_torch,
    random_state=42
)

train_dataset = TensorDataset(
    X_train_dl,
    y_train_dl
)

val_dataset = TensorDataset(
    X_val_dl,
    y_val_dl
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

model = EEGNet(n_classes=4)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

def accuracy(loader, model):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in loader:

            outputs = model(X_batch)

            preds = outputs.argmax(dim=1)

            correct += (preds == y_batch).sum().item()

            total += y_batch.size(0)

    return correct / total

num_epochs = 50

best_val_acc = 0

for epoch in range(num_epochs):

    model.train()

    running_loss = 0

    for X_batch, y_batch in train_loader:

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_acc = accuracy(train_loader, model)

    val_acc = accuracy(val_loader, model)

    epoch_loss = running_loss / len(train_loader)

    print(
        f"Epoch {epoch+1:2d} | "
        f"Loss {epoch_loss:.3f} | "
        f"Train {train_acc:.3f} | "
        f"Val {val_acc:.3f}"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        best_model_state = model.state_dict()

model.load_state_dict(best_model_state)

test_dataset = TensorDataset(
    x_test_torch,
    y_test_torch
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

test_acc = accuracy(
    test_loader,
    model
)

print("Test Accuracy:", test_acc)

Conclusion ： CSP + LDA is better than EEG-Net in a small dataset which the later one is easer to being overfitting

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Left hand", "Right hand", "Feet", "Tongue"]
)

disp.plot()
plt.show()

In [ ]:
model = EEGNet(n_classes=4)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

model.eval()

with torch.no_grad():
    outputs = model(X_train_dl[:16])

    loss = criterion(
        outputs,
        y_train_dl[:16]
    )

    preds = outputs.argmax(dim=1)

    acc = (preds == y_train_dl[:16]).float().mean()

print("Initial loss:", loss.item())
print("Initial accuracy:", acc.item())

import copy

num_epochs = 50

train_losses = []
train_accs = []
val_accs = []

best_val_acc = 0.0
best_model_state = None

for epoch in range(num_epochs):

    # =====================
    # Training
    # =====================
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    # =====================
    # Evaluation
    # =====================
    train_acc = accuracy(train_loader, model)
    val_acc = accuracy(val_loader, model)

    train_losses.append(epoch_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    # =====================
    # Save best model
    # =====================
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        # IMPORTANT:
        # deepcopy prevents the saved weights
        # from changing during later training
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch [{epoch+1:2d}/{num_epochs}] "
        f"| Loss: {epoch_loss:.4f} "
        f"| Train Acc: {train_acc:.3f} "
        f"| Val Acc: {val_acc:.3f}"
    )

model.load_state_dict(best_model_state)

print("Best validation accuracy:", best_val_acc)

test_acc = accuracy(test_loader, model)

print("Final EEGNet Test Accuracy:", test_acc)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

print(cm)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.colorbar()

plt.xticks(
    range(4),
    ["Left hand", "Right hand", "Feet", "Tongue"]
)

plt.yticks(
    range(4),
    ["Left hand", "Right hand", "Feet", "Tongue"]
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("EEGNet Confusion Matrix — Subject A01")

plt.show()